# 1 · Preprocessing — raw MiXCR TRB → one cloud per (patient, cell type)Turn raw MiXCR clonotype tables into clean per-repertoire clouds, one per (patient, cell type).Each `*TRB.tsv` is processed independently — nothing crosses files, cell types, or patients.**Design choice:** the CD4/CD8 overlap is **not** removed — each repertoire keeps its full clonotypeset, so the descriptor sees the real cloud. Labels come from FACS sorting (ground truth).**Expected output: 68 clouds, 37 patients.** The 6 MS patients split across folders (094, 099, 100,104, 107, 108) are merged by patient ID.

## ConfigEdit `DIRS` and `OUT_DIR` to your local checkout. Filters use UMI≥2 / reads≥2 (see step 3).

In [ ]:
import glob, os, refrom collections import defaultdictimport numpy as npimport pandas as pd# --- edit these paths ---DIRS = {    'T1D': '../raw/T1D_mixcr',    'MS':  '../raw/MS_mixcr',    'MS_HD': '../raw/MS_HD_mixcr',}OUT_DIR   = '../data/clouds_raw_TRB'MIN_READS = 2MIN_UMI   = 2AA = 'ACDEFGHIKLMNPQRSTVWY'_AA_RE = re.compile(f'^[{AA}]+$')os.makedirs(OUT_DIR, exist_ok=True)

## 1.1 · File census & identityParse (patient, cohort, cell type) from each filename and group files. Confirms we have the expected68 points before touching any data.

In [ ]:
def parse_identity(fname):    b = os.path.basename(fname)    m = re.match(r'(MS\d+|T1D\d+|HD_[A-Za-z]+)', b)    if not m:        return None    pid = m.group(1)    cohort = 'HD' if pid.startswith('HD') else ('T1D' if pid.startswith('T1D') else 'MS')    cell = 'CD4' if 'CD4' in b else ('CD8' if 'CD8' in b else None)    return (pid, cohort, cell) if cell else Nonefiles = []for d in DIRS.values():    files += glob.glob(os.path.join(d, '**', '*TRB.tsv'), recursive=True)files = sorted(set(files))groups = defaultdict(list)for f in files:    ident = parse_identity(f)    if ident:        groups[ident].append(f)print(f'{len(files)} TRB.tsv files')print(f'{len(groups)} (patient, cell_type) points')print(f'{len({p for (p,_,_) in groups})} unique patients')census = pd.Series({(c,ct): sum(1 for (p,cc,cct) in groups if cc==c and cct==ct)                    for c in ['HD','T1D','MS'] for ct in ['CD4','CD8']})print(census)

## 1.2 · Inspect one raw fileBefore filtering, look at the real shape of a MiXCR TSV and confirm the key columns:`readCount`, `uniqueMoleculeCount`, `nSeqCDR3`, `aaSeqCDR3`, `allVHitsWithScore`.

In [ ]:
example = files[0]raw = pd.read_csv(example, sep='\t', low_memory=False)print(os.path.basename(example), '->', raw.shape)print([c for c in raw.columns if c in       ['readCount','uniqueMoleculeCount','nSeqCDR3','aaSeqCDR3','allVHitsWithScore']])raw.head(3)

## 1.3 · Preprocessing functionsThe pipeline, step by step:- **Step 1** — PCR-error correction: within each (V gene, nt-length), merge clonotypes whose nt CDR3  differs by ≤1 base into the most abundant one (intra-file only).- **Step 2** — drop non-productive CDR3 (must be a clean amino-acid string).- **Step 3** — robustness filter: keep clonotypes with reads≥2 **and** UMI≥2. *On RNA data this can  remove a large fraction (the singletons) — worth watching.*- **Step 4** — CDR3 must start with C and be ≥8 aa.- **Step 6** — dedup to (cdr3aa, v_gene), summing counts; recompute frequency.Step 5 (pseudogene resolution) happens later, downstream, when the V gene can't be resolved.

In [ ]:
def first_gene(s):    s = str(s)    if not s or s.lower() == 'nan':        return None    g = s.split(',')[0].split('*')[0].split('(')[0].strip()    return g or Nonedef is_productive(s):    return isinstance(s, str) and bool(_AA_RE.match(s))def hamming1_merge(df):    """Step 1: within each (v_gene, nt-length), merge nt clonotypes differing by <=1 base    into the most abundant (readCount), summing counts. Intra-file only."""    if 'nSeqCDR3' not in df.columns:        return df    df = df.sort_values('readCount', ascending=False).reset_index(drop=True)    df['_len'] = df['nSeqCDR3'].str.len()    keep_rep = np.arange(len(df))    for (_v, _l), idx in df.groupby(['v_gene', '_len'], sort=False).groups.items():        idx = list(idx)        if len(idx) < 2:            continue        arr = [np.frombuffer(df.loc[r, 'nSeqCDR3'].encode(), dtype=np.uint8) for r in idx]        reps = []        for pos, r in enumerate(idx):            merged = False            for rep_row, rep_arr in reps:                if len(rep_arr) == len(arr[pos]) and int((rep_arr != arr[pos]).sum()) <= 1:                    keep_rep[r] = rep_row; merged = True; break            if not merged:                reps.append((r, arr[pos]))    df['_rep'] = keep_rep    agg = {'readCount': 'sum'}    if 'uniqueMoleculeCount' in df.columns:        agg['uniqueMoleculeCount'] = 'sum'    sums = df.groupby('_rep').agg(agg)    out = df[df.index.to_numpy() == df['_rep'].to_numpy()].copy()    out['readCount'] = out.index.map(sums['readCount'])    if 'uniqueMoleculeCount' in agg:        out['uniqueMoleculeCount'] = out.index.map(sums['uniqueMoleculeCount'])    return out.drop(columns=['_len', '_rep']).reset_index(drop=True)

## 1.4 · Run the funnel on the example fileHow many clonotypes survive each filter. **Watch step 3 (reads≥2 / UMI≥2):** on RNA data it can takeout a big fraction (the singletons).

In [ ]:
df = raw.copy()df['v_gene'] = df['allVHitsWithScore'].map(first_gene)df = df.rename(columns={'aaSeqCDR3': 'cdr3aa'}).dropna(subset=['v_gene', 'cdr3aa'])n_raw = len(df); print(f'raw ...................... {n_raw}')df = hamming1_merge(df)print(f'1. PCR correction ........ {len(df)}')df = df[df['cdr3aa'].map(is_productive)]print(f'2. productive ............ {len(df)}')if 'uniqueMoleculeCount' in df.columns:    df = df[(df['readCount'] >= MIN_READS) & (df['uniqueMoleculeCount'] >= MIN_UMI)]else:    df = df[df['readCount'] >= MIN_READS]print(f'3. robust >=2/>=2 ........ {len(df)}   <-- watch here')df = df[(df['cdr3aa'].str.startswith('C')) & (df['cdr3aa'].str.len() >= 8)]print(f'4. CDR3 (C.., >=8) ....... {len(df)}')df = df.groupby(['cdr3aa', 'v_gene'], as_index=False).agg(count=('readCount', 'sum'))df['freq'] = df['count'] / df['count'].sum()print(f'6. dedup (cdr3aa,v) ...... {len(df)}')print(f'   -> {100*len(df)/n_raw:.1f}% of raw kept')df.head()

## 1.5 · Process everything → one cloud per (patient, cell type)Same funnel wrapped in a function, run over all files. Multi-file patients are merged by ID.

In [ ]:
def preprocess_one(path):    df = pd.read_csv(path, sep='\t', low_memory=False)    df['v_gene'] = df['allVHitsWithScore'].map(first_gene)    df = df.rename(columns={'aaSeqCDR3': 'cdr3aa'}).dropna(subset=['v_gene', 'cdr3aa'])    df = hamming1_merge(df)    df = df[df['cdr3aa'].map(is_productive)]    if 'uniqueMoleculeCount' in df.columns:        df = df[(df['readCount'] >= MIN_READS) & (df['uniqueMoleculeCount'] >= MIN_UMI)]    else:        df = df[df['readCount'] >= MIN_READS]    df = df[(df['cdr3aa'].str.startswith('C')) & (df['cdr3aa'].str.len() >= 8)]    if df.empty:        return None    df = df.groupby(['cdr3aa', 'v_gene'], as_index=False).agg(count=('readCount', 'sum'))    df['freq'] = df['count'] / df['count'].sum()    return dfrows, written = [], 0for (pid, cohort, cell), fs in sorted(groups.items()):    parts = [preprocess_one(f) for f in fs]    parts = [p for p in parts if p is not None]    if not parts:        print(f'  {pid} {cell}: empty -> skip'); continue    cloud = (pd.concat(parts, ignore_index=True)               .groupby(['cdr3aa', 'v_gene'], as_index=False).agg(count=('count', 'sum')))    cloud['freq'] = cloud['count'] / cloud['count'].sum()    cloud['patient'], cloud['cohort'], cloud['cell_type'] = pid, cohort, cell    cloud.to_parquet(os.path.join(OUT_DIR, f'{pid}_{cell}_TRB.parquet'), index=False)    rows.append({'patient': pid, 'cohort': cohort, 'cell_type': cell, 'n_clonotypes': len(cloud)})    written += 1summary = pd.DataFrame(rows)print(f'\n{written} clouds written to {OUT_DIR}')

## 1.6 · Sanity check — depth and the CD4/CD8 confound**This is the key check.** If CD8 clouds systematically have far fewer clonotypes than CD4, thedescriptor could encode *depth* instead of *biology*. This number decides whether rarefaction isneeded downstream (it is — see the depth control in notebook 02).

In [ ]:
print('clouds per cohort x cell type:')print(summary.groupby(['cohort','cell_type']).size(), '\n')print('clonotypes per cloud (CD4 vs CD8):')print(summary.groupby('cell_type')['n_clonotypes'].describe()[['count','min','25%','50%','75%','max']])import matplotlib.pyplot as pltfig, ax = plt.subplots(figsize=(8, 5))for cell, col in [('CD4','#2C7FB8'), ('CD8','#C0392B')]:    ax.hist(summary[summary.cell_type==cell]['n_clonotypes'], bins=20, alpha=0.6,            label=cell, color=col, edgecolor='white')ax.set_xlabel('Clonotypes per cloud', fontsize=12, fontweight='bold')ax.set_ylabel('Number of clouds', fontsize=12, fontweight='bold')ax.set_title('Depth by cell type — is CD8 shallower? (the confound to control)', fontsize=12)ax.legend(fontsize=10); ax.grid(True, alpha=0.25)plt.tight_layout(); plt.show()